In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

import utils_data

In [ ]:
# Choose dataset to use
# For the MNIST dataset, we use the default torchvision data loader
# For YaleB Face dataset, download the data .zip and put it in the ./data folder. Since the authors have removed the original page, please download the "CroppedYale" from https://edstem.org/us/courses/84859/discussion/6955766

# DATASET = 'MNIST'
DATASET = 'YALEB'

X_all, y_all = utils_data.load_dataset(name=DATASET, root='./data')

# To make sure the data is loaded correctly, print unique values of y_all and their counts
print(np.unique(y_all, return_counts=True))

# For YALEB, you should see 
# (array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
#        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
#        34, 35, 36, 37]), array([65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 61, 60, 61, 64, 63, 64, 64,
#        65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65, 65,
#        65, 65, 65, 65]))

# For MNIST, you should see
# (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]))

In [ ]:
def filter_classes_or_individuals(classes):
    samples = []
    for c in classes:
        samples.append(X_all[y_all==c])
    return torch.vstack(samples)


## Filter out classes or individuals
## MNIST: Class 0 - 9
## YALEB: Class 0 - 37
# CLASSES = [37] # MODIFY THIS LINE
CLASSES = [0] # MODIFY THIS LINE
X = filter_classes_or_individuals(CLASSES)

# TODO: take the first xxx samples from X which will be useful for question (g)


# Visualize Samples
print(f'Number of samples: {X.shape[0]}')
sample_idxs = np.random.choice(X.shape[0], 10, replace=False)
fig, ax = plt.subplots(ncols=10, figsize=(30, 5))
for i, sample_idx in enumerate(sample_idxs):
    ax[i].imshow(X[sample_idx].reshape(28, 28), cmap='gray')
    ax[i].axis('off')
    ax[i].set_title(f'sample {sample_idx}')
plt.show()

## Part (a): Closed-form PPCA
Implement functions that compute the sample mean and covariance and return the closed-form PPCA estimates $(W^*, b^*, \sigma^*)$ given $X$ and latent dimension $d$. Use the formulas from class.

In [ ]:
# Closed-form PPCA helpers
n_samples, data_dim = X.shape

# Dimension parameter d of the latent variable 
latent_dim = 2

def compute_sample_mean_cov(X: torch.Tensor):
    # X: [N, D]
    mean = None # TODO: fill in
    cov = None # TODO: fill in
    return mean, cov

# implement closed-form PPCA (return W_star, b_star, sigma_star)
def closed_form_ppca(X: torch.Tensor, d: int):
    # TODO: compute W_star, b_star, sigma_star. You may find compute_sample_mean_cov useful.
    return W_star, b_star, sigma_star

# quick visualization of sample stats
sample_mean, sample_covariance = compute_sample_mean_cov(X)
print(sample_mean.shape, sample_covariance.shape)

fig, ax = plt.subplots(ncols=2)
ax[0].imshow(sample_mean.view(28, 28).numpy(), cmap='gray')
ax[0].set_title('sample mean')
ax[1].imshow(sample_covariance.numpy(), cmap='gray')
ax[1].set_title('sample covariance')
plt.show()

# Closed-form solution for PPCA 
W_star, b_star, sigma_star = closed_form_ppca(X, latent_dim)

## Part (b): EM for PPCA
Implement one EM iteration mapping $(b^k, W^k, \sigma^k)$ to $(b^{k+1}, W^{k+1}, \sigma^{k+1})$ using your analytical formulas in question 2 (b). Initialize with $b^0 =0$, $W^0 =$ `first d columns of I_D`, $\sigma^0 = 0.1$. Record the data log-likelihood per iteration for later plotting.

In [ ]:
# initialization is done for you
b_k = torch.zeros(data_dim)
W_k = torch.eye(data_dim)[:, :latent_dim].contiguous()
sigma_k = torch.tensor(0.1)

# implement E-step helper that returns Ez (Nxd) and EzzT (dxd)
# Ez is an N x d matrix whose j-th row is <z_j>^k defined in the equation above question 2(a)
# EzzT is the d x d matrix of \sum_{j=1}^N <z_j z_j^T>^k, defined in the equation in question 2(a) above the Hint 
def mstep_helper(X: torch.Tensor, b: torch.Tensor, W: torch.Tensor, sigma: torch.Tensor):
    # X: [N, D], b: [D], W: [D, d], sigma: scalar
    N, D = X.shape
    d = W.shape[1]
    # TODO: fill in
    return Ez, EzzT

# implement M-step update
# you should use the equations in Problem 2(b)
def mstep_update(X: torch.Tensor, Ez: torch.Tensor, EzzT: torch.Tensor):
    # TODO: fill in
    return b_next, W_next, sigma_next

# compute log-likelihood 
def log_likelihood(X: torch.Tensor, b: torch.Tensor, W: torch.Tensor, sigma: torch.Tensor):
    # p(x) = N(b, C) where C = W W^T + sigma^2 I
    N, D = X.shape
    C = W @ W.T + (sigma ** 2) * torch.eye(D, device=W.device, dtype=W.dtype)
    X_centered = X - b[None, :]
    # Use stable log-det and solve
    L = torch.linalg.cholesky(C)
    logdetC = 2.0 * torch.sum(torch.log(torch.diag(L)))
    # Solve C^{-1} X_centered^T via chol solve
    alpha = torch.cholesky_solve(X_centered.T, L)
    quad = torch.sum(X_centered.T * alpha)
    ll = -0.5 * (N * (D * torch.log(torch.tensor(2 * np.pi, dtype=W.dtype, device=W.device)) + logdetC) + quad)
    return ll


# Example skeleton for running and logging metrics
max_iter_em = 5000
em_ll_hist = []
kl_hist_em = []
subspace_hist_em = []
b_rel_hist_em = []
sigma_rel_hist_em = []
wall_time_em = []

# EM loop
import time
em_start = time.time()
for k_em in range(max_iter_em):
    Ez, EzzT = mstep_helper(X, b_k, W_k, sigma_k)
    b_k, W_k, sigma_k = mstep_update(X, Ez, EzzT)
    # Record the log-likelihood at each iteration so you can see if it increases over iterations.
    em_ll_hist.append(float(log_likelihood(X, b_k, W_k, sigma_k)))
    # TODO: in part (d), you will implement the metrics gaussian_kl and subspace_error, then uncomment the following lines
    # kl_hist_em.append(gaussian_kl(b_star, W_star @ W_star.T + sigma_star**2 * torch.eye(data_dim),
                                  b_k, W_k @ W_k.T + sigma_k**2 * torch.eye(data_dim)))
    # subspace_hist_em.append(subspace_error(W_k, W_star))
    b_rel_hist_em.append(float(torch.norm(b_star - b_k) / (torch.norm(b_star) + 1e-12)))
    sigma_rel_hist_em.append(float(torch.abs(sigma_k - sigma_star) / (torch.abs(sigma_star) + 1e-12)))
    wall_time_em.append(time.time() - em_start)

## Part (c): Gradient ascent for the PPCA ELBO
Implement gradient ascent on the PPCA ELBO with respect to $(W, b, \sigma)$ and $(A, c, \Sigma_e)$ with Adam optimizer. Try multiple learning rates and record the objective per iteration for later comparison.

In [ ]:
# max iteration 
max_iter_elbo = 5000
# adam learning rate
adam_lr = 1e-3

# TODO: implement elbo(X, W, b, sigma, A, c, Sigma_e) using analytical expression
# Since Sigma_e is a covairance matrix which must be positive-definite, we parameterize it as Sigma_e = E E^T

def elbo(X, W, b, sigma, A, c, E):
    N, D = X.shape
    d = W.shape[1]
    # TODO: fill in
    # you may find compute_sample_mean_cov useful. 
    return L

# Initialize model and variational params
W = torch.eye(data_dim)[:, :latent_dim].contiguous().clone().requires_grad_(True)
b = torch.zeros(data_dim, requires_grad=True)
sigma = torch.tensor(0.1, requires_grad=True)
A = torch.eye(data_dim)[:latent_dim, :].contiguous().clone().requires_grad_(True)
c = torch.zeros(latent_dim, requires_grad=True)
E = torch.eye(latent_dim).clone().requires_grad_(True)

# Example skeleton for running and logging metrics
elbo_ll_hist = []
kl_hist_elbo = []
subspace_hist_elbo = []
b_rel_hist_elbo = []
sigma_rel_hist_elbo = []
wall_time_elbo = []

# TODO: add an adam optimizer to optimize the encoder and decoder parameters with learning rate adam_lr
optimizer = None

elbo_start = time.time()
for k_elbo in range(max_iter_elbo):
    optimizer.zero_grad(set_to_none=True)
    L = elbo(X, W, b, sigma, A, c, E)

    # TODO: Here, you need to compute the loss, backpropagate (in the ascending direction of the loss), and step the optimizer 

    with torch.no_grad():
        sigma.clamp_(min=1e-6)

    # Record the log-likelihood at each iteration so you can see if it increases over iterations.
    elbo_ll_hist.append(log_likelihood(X, b, W, sigma).detach())

    # If you want to debug, you can print loss and log likelihood at each iteration
    # print(L.item(), log_likelihood(X, b, W, sigma).item())
    
    # TODO: part (d): Record other metrics (gaussian_kl, subspace_error, b relative error, sigma relative error), then uncomment the following lines
    # kl_hist_elbo.append(gaussian_kl(b_star, W_star @ W_star.T + sigma_star**2 * torch.eye(data_dim),
                                    b, W @ W.T + sigma**2 * torch.eye(data_dim)))
    # subspace_hist_elbo.append(subspace_error(W, W_star))
    b_rel_hist_elbo.append(float(torch.norm(b_star - b) / (torch.norm(b_star) + 1e-12)))
    sigma_rel_hist_elbo.append(float(torch.abs(sigma - sigma_star) / (torch.abs(sigma_star) + 1e-12)))
    wall_time_elbo.append(time.time() - elbo_start)

plt.figure()
plt.plot(elbo_ll_hist)
plt.xlabel('iteration')
plt.ylabel('ELBO')
plt.title('ELBO gradient ascent (Adam + Sigma_e=E E^T)')
plt.show()

## Sanity check
Below, one can evaluate the log likelihood function at the closed-form solution (i.e., a global maximizer), EM output, and ELBO output. We can also evaluate the ELBO variational objective function at the closed-form encoder-decoder parameters.  If the likelihood function, the closed-form solution, and the elbo functions are implemented correctly, we should observe true_ll = true_elbo. If EM and ELBO converge, then em_ll and elbo_ll should be close to true_ll. 


In [ ]:
true_ll = log_likelihood(X, b_star, W_star, sigma_star)
print("Optimal log likelihood (using closed-form solution): ", true_ll.item())

em_ll = log_likelihood(X, b_k, W_k, sigma_k)
print("EM log likelihood: ", em_ll.item())

elbo_ll = log_likelihood(X, b, W, sigma)
print("ELBO log likelihood: ", elbo_ll.item())

# variational parameters that optimize the ELBO
d = W_star.shape[1]
I_d = torch.eye(d, device=W_star.device, dtype=W_star.dtype)
M = W_star.T @ W_star + (sigma_star**2) * I_d
A_star = torch.linalg.inv(M) @ W_star.T
c_star = -A_star @ b_star
Sigma_e_star = (sigma_star**2) * torch.linalg.inv(M)
E_star = torch.linalg.cholesky(0.5 * (Sigma_e_star + Sigma_e_star.T) + 1e-6 * torch.eye(Sigma_e_star.shape[0], device=Sigma_e_star.device, dtype=Sigma_e_star.dtype))

true_elbo = elbo(X, W_star, b_star, sigma_star, A_star, c_star, E_star)
print("ELBO variational objective with optimal encoders and decoders: ", true_elbo.item())


## Another sanity check & Part (e): Sampling from a learned PPCA model
Part (e) includes sampling from a learned PPCA model. We put the code here for convenience, and further to allow you to visualize the learned PPCA models to see if they make sense. 

In [ ]:
# TODO: Fill in b_estimate, W_estimate, z, eps. You need to use the output of one of the methods (i.e., closed-form or EM or ELBO). 

# W_star, b_star, sigma_star

# sample and generate from latent space
X_sampled = []
N_SAMPLES = 10
# b_estimate = None # TODO: fill in 
# W_estimate = None # TODO: fill in 
for j in range(N_SAMPLES):
    # z = None # TODO: fill in 
    z = torch.randn(latent_dim, 1)
    # eps = None # TODO: fill in 
    eps = torch.randn(data_dim, 1) * sigma_star
    
    # generate sample for X
    # X_sampled_j = b_estimate[:, None] + W_estimate @ z + eps
    X_sampled_j = b_estimate[:, None] + W_estimate @ z
    X_sampled.append(X_sampled_j.view(28, 28))
    

# visualize sampled X
fig, ax = plt.subplots(ncols=10, figsize=(20, 3))
for j in range(N_SAMPLES):
    ax[j].imshow(X_sampled[j], cmap='gray')
    ax[j].axis('off')
plt.show()

## Part (d): Evaluation metrics and plots (MNIST and YaleB)

For EM and ELBO runs, produce plots with iteration on the x-axis and the following metrics on the y-axis (different curves per method):
- Log-likelihood
- Gaussian-to-Gaussian KL divergence to the closed-form fit
- W-subspace error (largest principal angle)
- b-relative error and sigma-relative error
- Wall-clock timing (per-iteration and cumulative)

Closed-form may be shown as a horizontal reference line where applicable.


In [ ]:
import time
from scipy.linalg import subspace_angles

def gaussian_kl(mu_p, Sigma_p, mu_q, Sigma_q):
    """KL divergence KL(N_p || N_q) for two Gaussians.
    """
    # TODO: fill in
    return float(kl.item())


def subspace_error(W_est, W_ref):
    """Largest principal angle in degrees between column-spaces of W_est and W_ref."""
    # TODO: fill in
    return float(np.degrees(largest))

# Feel free to change the plotting and scales to make the plots more informative and nice
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
axes[0].plot(em_ll_hist, label='EM')
axes[0].plot(elbo_ll_hist, label='ELBO')
axes[0].set_title('Objective')
axes[0].legend()

axes[1].plot(kl_hist_em, label='EM')
axes[1].plot(kl_hist_elbo, label='ELBO')
axes[1].set_title('KL to closed-form')
axes[1].legend()

axes[2].plot(subspace_hist_em, label='EM')
axes[2].plot(subspace_hist_elbo, label='ELBO')
axes[2].set_title('W subspace error (deg)')
axes[2].legend()

axes[3].plot(b_rel_hist_em, label='EM')
axes[3].plot(b_rel_hist_elbo, label='ELBO')
axes[3].set_title('b relative error')
axes[3].legend()

axes[4].plot(sigma_rel_hist_em, label='EM')
axes[4].plot(sigma_rel_hist_elbo, label='ELBO')
axes[4].set_title('sigma relative error')
axes[4].legend()

axes[5].plot(wall_time_em, label='EM')
axes[5].plot(wall_time_elbo, label='ELBO')
axes[5].set_title('Cumulative wall-clock (s)')
axes[5].legend()
plt.tight_layout()
plt.show()

## Part (e): Effect of noise level
For each method, use the estimated parameters to define a PPCA model, sample from it and visualize the samples. Vary the noise level and describe the quality of the image generation.

## Part (f): Effect of latent dimension $d$
Describe the quality of generation with respect to $d$.

## Part (g): Effect of sample size $N$
Keep $d$ fixed. Compare the performance when using many samples ($N \gg d$), a moderate number of samples ($N = d$), and a small number of samples $N < d$.

## Part (h)
Suppose now we move into a two/three-class regime, by adding classes to the *CLASSES* list. What do you observe about the sample mean and covariance? Based on the data samples we have, do the images generated make sense? 